# Compute condition numbers from CutFEM matrices in /build

In [ ]:
import numpy as np
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import splu, LinearOperator, onenormest

for i in range(3):
    # Read the data from the file
    mat = np.loadtxt("../../build/mat{}Cut.dat".format(i))
    # Extract row indices, column indices, and values
    row = mat[:, 0].astype(int)
    col = mat[:, 1].astype(int)
    data = mat[:, 2]
    # Adjust indices for zero-based indexing in Python
    row -= 1
    col -= 1
    # Create a sparse matrix in COO format and convert to CSR format
    A = coo_matrix((data, (row, col))).tocsr()
    # Estimate the 1-norm of the matrix A
    A1_norm = onenormest(A)
    try:
        # Perform LU decomposition of A
        lu = splu(A)
        n = A.shape[0]
        # Define matvec function for inv(A)
        def matvec(x):
            return lu.solve(x)
        # Define rmatvec function for inv(A)
        def rmatvec(x):
            return lu.solve(x, trans='T')
        # Create a LinearOperator representing inv(A)
        invA = LinearOperator((n, n), matvec=matvec, rmatvec=rmatvec, dtype=A.dtype)
        # Estimate the 1-norm of inv(A)
        invA1_norm_est = onenormest(invA)
        # Compute the condition number estimate
        cond_est = A1_norm * invA1_norm_est
        # Print the condition number in scientific notation with two decimal places
        print("{:.2E}".format(cond_est))
    except Exception as e:
        print(f"Failed to compute condition number estimate for matrix {i}: {e}")


/tmp/ipykernel_885639/3041535910.py:21: SparseEfficiencyWarning: splu converted its input to CSC format
  lu = splu(A)


1.36E+18
8.36E+16
